# Transformación de Datos - Práctica Semana 2

## Inteligencia de Negocios

## Grupo 9

### Integrantes

* Guevara Paz y Miño Juan Diego
* Moreno Jeremy
* Solis Paulo

---

## Descripción del proyecto

El presente proyecto se desarrolla en el contexto de un proceso ETL aplicado al análisis de datos relacionados con ciberseguridad global e indicadores digitales por país. Para ello, se utilizan tres fuentes de datos principales: incidentes globales de ciberseguridad, índices de ciberseguridad por país y estadísticas de usuarios de internet.

El objetivo del proyecto es preparar los datos para su posterior análisis mediante procesos de limpieza, transformación, normalización y validación. Estas actividades permiten mejorar la calidad de la información, reducir inconsistencias y facilitar la integración entre las fuentes utilizadas.

Durante esta práctica se trabajará principalmente en la etapa de transformación del proceso ETL. Se revisará la estructura de los datos, se identificarán valores nulos, duplicados e inconsistencias, se seleccionarán columnas relevantes, se aplicarán funciones de limpieza y se generarán datos transformados que puedan ser utilizados para análisis posteriores o carga en una base de datos.

El dominio seleccionado, ciberseguridad, permite relacionar información sobre tipos de ataques, pérdidas financieras, usuarios afectados, mecanismos de defensa, indicadores de madurez digital y penetración de internet. Esto aporta una base útil para comprender tendencias, riesgos y oportunidades de mejora en la gestión de la seguridad de la información.

## 1. Exploración inicial de las fuentes de datos

Antes de aplicar procesos de limpieza y transformación, se realiza una exploración inicial de las tres fuentes de datos utilizadas en el proyecto. Esta revisión permite comprender la estructura general de cada archivo, identificar la cantidad de filas y columnas, revisar los tipos de datos, detectar valores nulos, registros duplicados, columnas relevantes y posibles campos de relación entre las fuentes.

Las fuentes analizadas son:

* **Global Cybersecurity Threats 2015-2024:** contiene información sobre incidentes globales de ciberseguridad, incluyendo país, año, tipo de ataque, industria afectada, pérdidas financieras, usuarios afectados, fuente del ataque, vulnerabilidad explotada, mecanismo de defensa y tiempo de resolución.
* **Cyber Security Index:** contiene indicadores de ciberseguridad por país, como CEI, GCI, NCSI y DDL.
* **Internet Users by Country:** contiene información sobre usuarios de internet, población, región, subregión, año y porcentaje de usuarios de internet por país.

Esta fase es importante porque permite identificar problemas de calidad antes de transformar los datos. De esta forma, se pueden tomar decisiones adecuadas sobre limpieza, normalización y selección de columnas para el análisis posterior.

In [129]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# Configuración para visualizar mejor los resultados en el notebook
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Ruta base de los archivos CSV
DATA_DIR = Path("data")

# Rutas de las tres fuentes de datos
ruta_threats = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
ruta_cyber_security = DATA_DIR / "Cyber_security.csv"
ruta_internet_users = DATA_DIR / "internet_users_by_country_cleaned.csv"

# Validación básica de existencia de archivos antes de cargarlos
rutas = [ruta_threats, ruta_cyber_security, ruta_internet_users]
archivos_faltantes = [str(ruta) for ruta in rutas if not ruta.exists()]

if archivos_faltantes:
    raise FileNotFoundError(
        "No se encontraron los siguientes archivos CSV. "
        "Verifica que la carpeta 'data' esté en la misma ubicación del notebook: "
        + ", ".join(archivos_faltantes)
    )

# Carga de las tres fuentes de datos
df_threats = pd.read_csv(ruta_threats)
df_cyber_security = pd.read_csv(ruta_cyber_security)
df_internet_users = pd.read_csv(ruta_internet_users)

print("Fuentes de datos cargadas correctamente.")

Fuentes de datos cargadas correctamente.


In [130]:
# Crear un diccionario para manejar las fuentes de forma organizada
datasets = {
    "Global Cybersecurity Threats": df_threats,
    "Cyber Security Index": df_cyber_security,
    "Internet Users by Country": df_internet_users
}

# Resumen general de filas, columnas, duplicados y valores nulos
resumen_general = []

for nombre, df in datasets.items():
    resumen_general.append({
        "fuente": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "registros_duplicados": int(df.duplicated().sum()),
        "total_valores_nulos": int(df.isnull().sum().sum())
    })

df_resumen_general = pd.DataFrame(resumen_general)
df_resumen_general

,fuente,filas,columnas,registros_duplicados,total_valores_nulos
0,Global Cybersecurity Threats,3000,10,0,0
1,Cyber Security Index,192,6,0,151
2,Internet Users by Country,215,7,0,0


In [131]:
# Revisión de columnas, tipos de datos, valores nulos y valores únicos por cada fuente
for nombre, df in datasets.items():
    print(f"\nFuente: {nombre}")
    display(pd.DataFrame({
        "columna": df.columns,
        "tipo_dato": df.dtypes.astype(str).values,
        "valores_nulos": df.isnull().sum().values,
        "porcentaje_nulos": (df.isnull().mean().values * 100).round(2),
        "valores_unicos": df.nunique().values
    }))


Fuente: Global Cybersecurity Threats


,columna,tipo_dato,valores_nulos,porcentaje_nulos,valores_unicos
0,Country,str,0,0.0,10
1,Year,int64,0,0.0,10
2,Attack Type,str,0,0.0,6
3,Target Industry,str,0,0.0,7
4,Financial Loss (in Million $),float64,0,0.0,2536
5,Number of Affected Users,int64,0,0.0,2998
6,Attack Source,str,0,0.0,4
7,Security Vulnerability Type,str,0,0.0,4
8,Defense Mechanism Used,str,0,0.0,5
9,Incident Resolution Time (in Hours),int64,0,0.0,72



Fuente: Cyber Security Index


,columna,tipo_dato,valores_nulos,porcentaje_nulos,valores_unicos
0,Country,str,0,0.00,192
1,Region,str,0,0.00,5
2,CEI,float64,84,43.75,85
3,GCI,float64,2,1.04,180
4,NCSI,float64,25,13.02,69
5,DDL,float64,40,20.83,150



Fuente: Internet Users by Country


,columna,tipo_dato,valores_nulos,porcentaje_nulos,valores_unicos
0,Country or area,str,0,0.0,215
1,Subregion,str,0,0.0,22
2,Region,str,0,0.0,5
3,Internet users,int64,0,0.0,215
4,Population_2021,int64,0,0.0,215
5,Year,int64,0,0.0,5
6,Percentage_Internet_Users,float64,0,0.0,195


In [132]:
# Mostrar las primeras filas de cada fuente para revisar su estructura visualmente
for nombre, df in datasets.items():
    print(f"\nVista preliminar: {nombre}")
    display(df.head())


Vista preliminar: Global Cybersecurity Threats


,Country,Year,Attack Type,Target Industry,Financial Loss (in Million $),Number of Affected Users,Attack Source,Security Vulnerability Type,Defense Mechanism Used,Incident Resolution Time (in Hours)
0,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63
1,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71
2,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20
3,UK,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7
4,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68



Vista preliminar: Cyber Security Index


,Country,Region,CEI,GCI,NCSI,DDL
0,Afghanistan,Asia-Pasific,1.000,5.20,11.69,19.50
1,Albania,Europe,0.566,64.32,62.34,48.74
2,Algeria,Africa,0.721,33.95,33.77,42.81
3,Andorra,Europe,NaN,26.38,NaN,NaN
4,Angola,Africa,NaN,12.99,9.09,22.69



Vista preliminar: Internet Users by Country


,Country or area,Subregion,Region,Internet users,Population_2021,Year,Percentage_Internet_Users
0,China,Eastern Asia,Asia,1102140000,1425893465,2022,77.3
1,India,Southern Asia,Asia,881250000,1407563842,2024,62.6
2,United States,Northern America,Americas,311300000,336997624,2023,92.4
3,Indonesia,South-eastern Asia,Asia,215626156,273753191,2023,78.8
4,Pakistan,Southern Asia,Asia,170000000,240000000,2022,70.8


In [133]:
# Revisión específica de registros duplicados
duplicados = []

for nombre, df in datasets.items():
    duplicados.append({
        "fuente": nombre,
        "cantidad_duplicados": int(df.duplicated().sum())
    })

df_duplicados = pd.DataFrame(duplicados)
df_duplicados

,fuente,cantidad_duplicados
0,Global Cybersecurity Threats,0
1,Cyber Security Index,0
2,Internet Users by Country,0


In [134]:
from pandas.api.types import is_object_dtype, is_string_dtype

def revisar_columnas_texto(df, nombre_fuente):
    """
    Revisa columnas de tipo texto para identificar posibles problemas de formato,
    como espacios al inicio o final, textos vacíos y cantidad de valores únicos.
    """

    # Seleccionar columnas de texto de forma compatible con versiones actuales y futuras de pandas
    columnas_texto = [
        columna for columna in df.columns
        if is_object_dtype(df[columna]) or is_string_dtype(df[columna])
    ]

    resultados = []

    for columna in columnas_texto:
        serie = df[columna].astype("string")

        resultados.append({
            "fuente": nombre_fuente,
            "columna": columna,
            "valores_unicos": int(df[columna].nunique()),
            "valores_con_espacios": int((serie != serie.str.strip()).sum()),
            "valores_vacios": int(serie.str.strip().eq("").sum())
        })

    return pd.DataFrame(resultados)


revision_texto = pd.concat(
    [revisar_columnas_texto(df, nombre) for nombre, df in datasets.items()],
    ignore_index=True
)

revision_texto

,fuente,columna,valores_unicos,valores_con_espacios,valores_vacios
0,Global Cybersecurity Threats,Country,10,0,0
1,Global Cybersecurity Threats,Attack Type,6,0,0
2,Global Cybersecurity Threats,Target Industry,7,0,0
3,Global Cybersecurity Threats,Attack Source,4,0,0
4,Global Cybersecurity Threats,Security Vulnerability Type,4,0,0
5,Global Cybersecurity Threats,Defense Mechanism Used,5,0,0
6,Cyber Security Index,Country,192,0,0
7,Cyber Security Index,Region,5,0,0
8,Internet Users by Country,Country or area,215,0,0
9,Internet Users by Country,Subregion,22,0,0


In [135]:
# Revisión de columnas que podrían servir para relacionar las fuentes
columnas_relacion = pd.DataFrame({
    "campo": ["Country / Country or area", "Year", "Region"],
    "presente_en": [
        "Global Cybersecurity Threats, Cyber Security Index, Internet Users by Country",
        "Global Cybersecurity Threats, Internet Users by Country",
        "Cyber Security Index, Internet Users by Country"
    ],
    "uso_potencial": [
        "Permite relacionar las tres fuentes a nivel de país, previa normalización de nombres.",
        "Permite analizar datos por año, aunque debe validarse porque no todas las fuentes manejan el mismo rango temporal.",
        "Permite realizar análisis regionales, siempre que los nombres de regiones estén homologados."
    ]
})

columnas_relacion

,campo,presente_en,uso_potencial
0,Country / Country or area,"Global Cybersecurity Threats, Cyber Security I...",Permite relacionar las tres fuentes a nivel de...
1,Year,"Global Cybersecurity Threats, Internet Users b...","Permite analizar datos por año, aunque debe va..."
2,Region,"Cyber Security Index, Internet Users by Country","Permite realizar análisis regionales, siempre ..."


In [136]:
# Revisión de países por fuente
print("Cantidad de países en Global Cybersecurity Threats:", df_threats["Country"].nunique())
print("Cantidad de países en Cyber Security Index:", df_cyber_security["Country"].nunique())
print("Cantidad de países en Internet Users by Country:", df_internet_users["Country or area"].nunique())

# Países presentes en el dataset de amenazas globales
paises_threats = sorted(df_threats["Country"].unique())

print("\nPaíses presentes en Global Cybersecurity Threats:")
for pais in paises_threats:
    print("-", pais)

# Revisión de regiones en las fuentes que tienen columna Region
print("\nRegiones en Cyber Security Index:")
display(df_cyber_security["Region"].value_counts().rename_axis("region").reset_index(name="cantidad"))

print("\nRegiones en Internet Users by Country:")
display(df_internet_users["Region"].value_counts().rename_axis("region").reset_index(name="cantidad"))

Cantidad de países en Global Cybersecurity Threats: 10
Cantidad de países en Cyber Security Index: 192
Cantidad de países en Internet Users by Country: 215

Países presentes en Global Cybersecurity Threats:
- Australia
- Brazil
- China
- France
- Germany
- India
- Japan
- Russia
- UK
- USA

Regiones en Cyber Security Index:


,region,cantidad
0,Asia-Pasific,56
1,Africa,53
2,Europe,48
3,North America,24
4,South America,11



Regiones en Internet Users by Country:


,region,cantidad
0,Africa,55
1,Asia,49
2,Europe,47
3,Americas,45
4,Oceania,19


In [137]:
# Homologación preliminar de nombres de países para comparación
country_mapping = {
    "USA": "United States",
    "UK": "United Kingdom"
}

paises_threats_normalizados = set(
    df_threats["Country"]
    .replace(country_mapping)
    .str.strip()
)

paises_cyber_security = set(
    df_cyber_security["Country"]
    .str.strip()
)

paises_internet_users = set(
    df_internet_users["Country or area"]
    .str.strip()
)

# Validar coincidencias entre fuentes
coinciden_cyber_security = paises_threats_normalizados.intersection(paises_cyber_security)
coinciden_internet_users = paises_threats_normalizados.intersection(paises_internet_users)
coinciden_todas = paises_threats_normalizados.intersection(paises_cyber_security).intersection(paises_internet_users)

print("Países del dataset de amenazas que coinciden con Cyber Security Index:", len(coinciden_cyber_security))
print(sorted(coinciden_cyber_security))

print("\nPaíses del dataset de amenazas que coinciden con Internet Users:", len(coinciden_internet_users))
print(sorted(coinciden_internet_users))

print("\nPaíses que coinciden en las tres fuentes:", len(coinciden_todas))
print(sorted(coinciden_todas))

Países del dataset de amenazas que coinciden con Cyber Security Index: 10
['Australia', 'Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'Russia', 'United Kingdom', 'United States']

Países del dataset de amenazas que coinciden con Internet Users: 10
['Australia', 'Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'Russia', 'United Kingdom', 'United States']

Países que coinciden en las tres fuentes: 10
['Australia', 'Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'Russia', 'United Kingdom', 'United States']


In [138]:
# Selección preliminar de columnas relevantes para el proyecto
columnas_relevantes = {
    "Global Cybersecurity Threats": [
        "Country",
        "Year",
        "Attack Type",
        "Target Industry",
        "Financial Loss (in Million $)",
        "Number of Affected Users",
        "Attack Source",
        "Security Vulnerability Type",
        "Defense Mechanism Used",
        "Incident Resolution Time (in Hours)"
    ],
    "Cyber Security Index": [
        "Country",
        "Region",
        "CEI",
        "GCI",
        "NCSI",
        "DDL"
    ],
    "Internet Users by Country": [
        "Country or area",
        "Subregion",
        "Region",
        "Internet users",
        "Population_2021",
        "Year",
        "Percentage_Internet_Users"
    ]
}

for fuente, columnas in columnas_relevantes.items():
    print(f"\nColumnas relevantes para {fuente}:")
    for columna in columnas:
        print(f"- {columna}")


Columnas relevantes para Global Cybersecurity Threats:
- Country
- Year
- Attack Type
- Target Industry
- Financial Loss (in Million $)
- Number of Affected Users
- Attack Source
- Security Vulnerability Type
- Defense Mechanism Used
- Incident Resolution Time (in Hours)

Columnas relevantes para Cyber Security Index:
- Country
- Region
- CEI
- GCI
- NCSI
- DDL

Columnas relevantes para Internet Users by Country:
- Country or area
- Subregion
- Region
- Internet users
- Population_2021
- Year
- Percentage_Internet_Users


### Hallazgos de la exploración inicial

Después de revisar las tres fuentes de datos, se identificaron los siguientes hallazgos:

#### Global Cybersecurity Threats 2015-2024

Esta fuente contiene información detallada sobre incidentes globales de ciberseguridad. Está compuesta por registros asociados a países, años, tipos de ataque, industrias afectadas, pérdidas financieras, usuarios afectados, vulnerabilidades, mecanismos de defensa y tiempo de resolución.

Durante la exploración inicial se observa que esta fuente cuenta con 3000 registros y 10 columnas. No presenta valores nulos en sus columnas principales y no contiene registros duplicados. Sus columnas son relevantes para el proyecto porque permiten analizar el comportamiento de los ataques, el impacto económico, la cantidad de usuarios afectados y los tiempos de respuesta ante incidentes.

La columna **Country** puede utilizarse como campo de relación con las otras fuentes, siempre que previamente se normalicen los nombres de países.

#### Cyber Security Index

Esta fuente contiene indicadores de ciberseguridad por país, tales como CEI, GCI, NCSI y DDL. Estos indicadores permiten complementar el análisis de incidentes con variables relacionadas con exposición, madurez y desarrollo digital.

En esta fuente se identifican 192 registros y 6 columnas. También se observan valores nulos en algunas columnas numéricas, principalmente en los indicadores CEI, GCI, NCSI y DDL. Estos valores deberán ser tratados en la fase de limpieza, ya sea mediante imputación, marcado de datos incompletos o conservación controlada según el objetivo del análisis.

También se observa que la columna **Country** puede funcionar como clave de relación con las otras fuentes. Además, la columna **Region** puede ser útil para análisis agrupados por zona geográfica.

#### Internet Users by Country

Esta fuente contiene datos sobre usuarios de internet, población, región, subregión, año y porcentaje de usuarios de internet por país. Su información es importante porque permite contextualizar los incidentes de ciberseguridad en función del nivel de conectividad digital de cada país.

Durante la exploración inicial se identifican 215 registros y 7 columnas. No se evidencian valores nulos ni registros duplicados. Las columnas más relevantes son **Country or area**, **Internet users**, **Population_2021**, **Year** y **Percentage_Internet_Users**.

La columna **Country or area** deberá ser normalizada para poder relacionarse correctamente con la columna **Country** de las otras fuentes. También se debe revisar la columna **Year**, ya que puede ayudar en ciertos análisis temporales, pero no todas las fuentes manejan el mismo rango de años.

#### Posibles columnas de relación

La principal columna de relación entre las tres fuentes es el país. En el dataset de amenazas aparece como **Country**, en el índice de ciberseguridad también como **Country**, y en el dataset de usuarios de internet como **Country or area**.

También existen campos como **Year** y **Region**, pero deben utilizarse con cuidado. El campo **Year** no está presente en todas las fuentes, y las regiones no manejan exactamente las mismas categorías entre datasets.

#### Hallazgo sobre países y regiones

Al revisar la cantidad de países presentes en cada fuente, se observa que el dataset **Global Cybersecurity Threats** contiene únicamente 10 países, mientras que **Cyber Security Index** contiene 192 países y **Internet Users by Country** contiene 215 países o áreas geográficas.

Esta diferencia no representa un error, sino una característica de las fuentes utilizadas. El dataset de amenazas está enfocado en incidentes registrados para un conjunto reducido de países, mientras que las otras dos fuentes tienen una cobertura global más amplia.

Para poder relacionar las tres fuentes, se identifica que la columna principal de unión será el país. Sin embargo, antes de realizar cualquier integración es necesario homologar ciertos nombres, por ejemplo **USA** a **United States** y **UK** a **United Kingdom**.

Después de aplicar esta homologación preliminar, los 10 países presentes en el dataset de amenazas pueden relacionarse con las otras dos fuentes. Esto confirma que es posible integrar los datasets usando el país como campo principal de relación.

También se observa que las regiones no están escritas exactamente igual entre las fuentes. Por ejemplo, en **Cyber Security Index** aparece la región **Asia-Pasific**, mientras que en **Internet Users by Country** se manejan regiones como **Asia**, **Africa**, **Europe**, **Americas** y **Oceania**. Por esta razón, las regiones deberán revisarse y normalizarse si se desean utilizar para análisis comparativos.

En conclusión, las tres fuentes son útiles para el proyecto, pero requieren procesos de limpieza y normalización antes de integrarse o utilizarse en análisis posteriores.


## 2. Configuración segura del proyecto

En esta fase se configura el uso de variables de entorno para proteger información sensible del proyecto, especialmente los parámetros de conexión a PostgreSQL. En lugar de escribir credenciales directamente dentro del notebook, se utiliza un archivo `.env`, el cual permite centralizar la configuración y evitar que datos sensibles queden expuestos en el código.

Las variables de entorno utilizadas en este proyecto son:

- **POSTGRES_USER:** usuario de conexión a PostgreSQL.
- **POSTGRES_PASSWORD:** contraseña del usuario de PostgreSQL.
- **POSTGRES_DB:** nombre de la base de datos utilizada para el proyecto ETL.
- **POSTGRES_HOST:** servidor o host donde se encuentra PostgreSQL.
- **POSTGRES_PORT:** puerto de conexión al servicio de PostgreSQL.

El uso de variables de entorno es importante porque separa la configuración sensible de la lógica del programa. Esto mejora la seguridad, facilita el mantenimiento del proyecto y permite ejecutar el notebook en diferentes ambientes sin modificar directamente el código fuente.

Además, esta práctica aporta a la seguridad del proyecto porque reduce el riesgo de exponer credenciales en repositorios, capturas de pantalla, entregas académicas o archivos compartidos. En un entorno profesional, este enfoque es especialmente relevante para proteger accesos a bases de datos, servicios externos, tokens, rutas privadas o parámetros críticos de infraestructura.

In [139]:
from dotenv import load_dotenv
import os

# Cargar variables de entorno desde el archivo .env ubicado en la carpeta del proyecto
ENV_PATH = Path(".env")

if not ENV_PATH.exists():
    raise FileNotFoundError(
        "No se encontró el archivo .env. Verifica que esté en la misma carpeta del notebook."
    )

load_dotenv(dotenv_path=ENV_PATH)

# Lectura de variables de conexión a PostgreSQL
DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME = os.getenv("POSTGRES_DB")
DB_HOST = os.getenv("POSTGRES_HOST")
DB_PORT = os.getenv("POSTGRES_PORT")

# Validar que las variables requeridas existan
variables_requeridas = {
    "POSTGRES_USER": DB_USER,
    "POSTGRES_PASSWORD": DB_PASSWORD,
    "POSTGRES_DB": DB_NAME,
    "POSTGRES_HOST": DB_HOST,
    "POSTGRES_PORT": DB_PORT
}

variables_faltantes = [nombre for nombre, valor in variables_requeridas.items() if not valor]

if variables_faltantes:
    raise EnvironmentError(
        "Faltan variables de entorno requeridas en el archivo .env: "
        + ", ".join(variables_faltantes)
    )

# Mostrar configuración sin exponer información sensible
configuracion_segura = pd.DataFrame({
    "variable": [
        "POSTGRES_USER",
        "POSTGRES_PASSWORD",
        "POSTGRES_DB",
        "POSTGRES_HOST",
        "POSTGRES_PORT"
    ],
    "valor_cargado": [
        DB_USER,
        "********",
        DB_NAME,
        DB_HOST,
        DB_PORT
    ],
    "descripcion": [
        "Usuario de conexión a PostgreSQL",
        "Contraseña protegida y no visible en el notebook",
        "Nombre de la base de datos del proyecto",
        "Host o servidor donde se ejecuta PostgreSQL",
        "Puerto expuesto para la conexión a PostgreSQL"
    ]
})

configuracion_segura

,variable,valor_cargado,descripcion
0,POSTGRES_USER,admin,Usuario de conexión a PostgreSQL
1,POSTGRES_PASSWORD,********,Contraseña protegida y no visible en el notebook
2,POSTGRES_DB,db_ciberseguridad_etl,Nombre de la base de datos del proyecto
3,POSTGRES_HOST,localhost,Host o servidor donde se ejecuta PostgreSQL
4,POSTGRES_PORT,5432,Puerto expuesto para la conexión a PostgreSQL


In [140]:
try:
    from sqlalchemy import create_engine, text

    # Construcción de la cadena de conexión usando variables de entorno
    connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

    # Creación del engine de conexión a PostgreSQL
    engine = create_engine(connection_string)

    try:
        with engine.connect() as connection:
            resultado = connection.execute(text("SELECT current_database(), current_user;"))
            database_name, database_user = resultado.fetchone()

            print("Conexión exitosa a PostgreSQL")
            print("Base de datos conectada:", database_name)
            print("Usuario conectado:", database_user)
    except Exception as e:
        print("Error al conectar con PostgreSQL:")
        print(e)

except ModuleNotFoundError:
    print("No se encontró SQLAlchemy en el entorno actual.")
    print("Instala las dependencias con: pip install sqlalchemy psycopg2-binary")

Conexión exitosa a PostgreSQL
Base de datos conectada: db_ciberseguridad_etl
Usuario conectado: admin


### Hallazgos de la configuración segura

La configuración segura del proyecto se realizó mediante el uso de variables de entorno almacenadas en un archivo `.env`. Estas variables permiten manejar los datos de conexión a PostgreSQL sin escribir directamente credenciales sensibles dentro del notebook.

Durante esta fase se cargaron variables como el usuario, contraseña, nombre de la base de datos, host y puerto de conexión. Para evitar exposición de información sensible, la contraseña no se imprime en pantalla y se muestra únicamente como un valor enmascarado.

Esta práctica mejora la seguridad del proyecto porque separa las credenciales de la lógica del código. Además, facilita que el mismo notebook pueda ejecutarse en otros ambientes, cambiando únicamente el archivo `.env` y no el código fuente.

En un contexto profesional, este enfoque es importante porque ayuda a prevenir la filtración accidental de credenciales en repositorios, documentos, entregas, capturas de pantalla o ambientes compartidos. También permite mantener una mejor organización y control sobre los parámetros de conexión utilizados por el proceso ETL.

## 3. Limpieza de datos

En esta fase se aplican funciones de limpieza sobre las tres fuentes del proyecto. El objetivo es preparar los datos para las siguientes etapas del proceso ETL, corrigiendo nombres de columnas, normalizando textos, eliminando duplicados, convirtiendo tipos de datos, tratando valores nulos y limpiando las columnas que podrían utilizarse como claves de relación.

Para mantener el código organizado, se definen funciones reutilizables que permiten aplicar reglas similares a diferentes DataFrames. Esto facilita la trazabilidad del proceso y evita repetir instrucciones de limpieza en varias partes del notebook.

In [141]:
import re
import unicodedata

# Diccionario para homologar nombres de países que aparecen abreviados o con escrituras distintas.
# Esta limpieza es importante porque el país será una de las principales columnas de relación.
COUNTRY_MAPPING = {
    "USA": "United States",
    "U.S.A.": "United States",
    "US": "United States",
    "UK": "United Kingdom",
    "U.K.": "United Kingdom"
}


def normalizar_nombre_columna(columna):
    """
    Convierte nombres de columnas a formato snake_case.
    Ejemplo: 'Financial Loss (in Million $)' -> 'financial_loss_in_million'
    """
    columna = columna.strip().lower()
    columna = columna.replace("$", "")
    columna = re.sub(r"[()/%]+", " ", columna)
    columna = re.sub(r"[^a-z0-9]+", "_", columna)
    columna = re.sub(r"_+", "_", columna)
    columna = columna.strip("_")
    return columna


def normalizar_texto(valor):
    """
    Limpia valores de texto eliminando espacios innecesarios.
    Si el valor está vacío o es nulo, devuelve pd.NA.
    """
    if pd.isna(valor):
        return pd.NA
    
    valor = str(valor).strip()
    valor = re.sub(r"\s+", " ", valor)
    
    if valor == "":
        return pd.NA
    
    return valor


def homologar_pais(valor):
    """
    Normaliza y homologa nombres de países para facilitar la relación entre fuentes.
    """
    valor = normalizar_texto(valor)
    
    if pd.isna(valor):
        return pd.NA
    
    return COUNTRY_MAPPING.get(valor, valor)


def limpiar_columnas_texto(df):
    """
    Aplica limpieza básica a todas las columnas de texto de un DataFrame.
    """
    df_limpio = df.copy()
    
    columnas_texto = df_limpio.select_dtypes(include=["object", "string"]).columns
    
    for columna in columnas_texto:
        df_limpio[columna] = df_limpio[columna].apply(normalizar_texto)
    
    return df_limpio


def convertir_columnas_numericas(df, columnas):
    """
    Convierte columnas específicas a valores numéricos.
    Los valores no convertibles se transforman en NaN para ser tratados posteriormente.
    """
    df_limpio = df.copy()
    
    for columna in columnas:
        if columna in df_limpio.columns:
            df_limpio[columna] = pd.to_numeric(df_limpio[columna], errors="coerce")
    
    return df_limpio


def resumen_limpieza(nombre_fuente, df_original, df_limpio):
    """
    Genera un resumen comparativo antes y después de la limpieza.
    """
    return {
        "fuente": nombre_fuente,
        "filas_originales": df_original.shape[0],
        "filas_limpias": df_limpio.shape[0],
        "columnas_originales": df_original.shape[1],
        "columnas_limpias": df_limpio.shape[1],
        "duplicados_originales": df_original.duplicated().sum(),
        "duplicados_limpios": df_limpio.duplicated().sum(),
        "nulos_originales": df_original.isnull().sum().sum(),
        "nulos_limpios": df_limpio.isnull().sum().sum()
    }

print("Funciones generales de limpieza creadas correctamente.")

Funciones generales de limpieza creadas correctamente.


### 3.1 Limpieza del dataset Global Cybersecurity Threats

En esta fuente se limpian los nombres de columnas, se eliminan duplicados, se normalizan las columnas de texto y se homologa la columna de país. También se convierten las columnas numéricas principales para asegurar que puedan utilizarse en cálculos posteriores.

Las columnas numéricas más importantes son el año, la pérdida financiera, el número de usuarios afectados y el tiempo de resolución del incidente.

In [142]:
def limpiar_threats(df):
    """
    Limpia el DataFrame de amenazas globales de ciberseguridad.
    """
    df_limpio = df.copy()
    
    # Corrección de nombres de columnas a formato snake_case
    df_limpio.columns = [normalizar_nombre_columna(col) for col in df_limpio.columns]
    
    # Eliminación de registros duplicados
    df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
    
    # Normalización de textos
    df_limpio = limpiar_columnas_texto(df_limpio)
    
    # Homologación de país como posible clave de relación
    df_limpio["country"] = df_limpio["country"].apply(homologar_pais)
    
    # Conversión de columnas numéricas
    columnas_numericas = [
        "year",
        "financial_loss_in_million",
        "number_of_affected_users",
        "incident_resolution_time_in_hours"
    ]
    df_limpio = convertir_columnas_numericas(df_limpio, columnas_numericas)
    
    # Conversión del año a entero nullable para conservar consistencia si existieran valores faltantes
    df_limpio["year"] = df_limpio["year"].astype("Int64")
    
    return df_limpio


df_threats_clean = limpiar_threats(df_threats)

df_threats_clean.head()

,country,year,attack_type,target_industry,financial_loss_in_million,number_of_affected_users,attack_source,security_vulnerability_type,defense_mechanism_used,incident_resolution_time_in_hours
0,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63
1,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71
2,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20
3,United Kingdom,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7
4,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68


### 3.2 Limpieza del dataset Cyber Security Index

En esta fuente se limpian los nombres de columnas, se eliminan duplicados, se normalizan los textos y se homologa el país. Además, se corrige la categoría regional **Asia-Pasific** a **Asia-Pacific**, ya que presenta un problema de escritura.

Los indicadores CEI, GCI, NCSI y DDL se convierten a tipo numérico. Los valores nulos se conservan inicialmente para no alterar el significado de los indicadores, pero se agrega una columna auxiliar que identifica si el registro tiene indicadores incompletos.

In [143]:
def limpiar_cyber_security(df):
    """
    Limpia el DataFrame de índices de ciberseguridad por país.
    """
    df_limpio = df.copy()
    
    # Corrección de nombres de columnas
    df_limpio.columns = [normalizar_nombre_columna(col) for col in df_limpio.columns]
    
    # Eliminación de duplicados
    df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
    
    # Normalización de textos
    df_limpio = limpiar_columnas_texto(df_limpio)
    
    # Homologación de país
    df_limpio["country"] = df_limpio["country"].apply(homologar_pais)
    
    # Estandarización de categorías de región
    df_limpio["region"] = df_limpio["region"].replace({
        "Asia-Pasific": "Asia-Pacific"
    })
    
    # Conversión de indicadores a tipo numérico
    columnas_numericas = ["cei", "gci", "ncsi", "ddl"]
    df_limpio = convertir_columnas_numericas(df_limpio, columnas_numericas)
    
    # Columna auxiliar para identificar registros con indicadores incompletos
    df_limpio["has_missing_security_index"] = df_limpio[columnas_numericas].isnull().any(axis=1)
    
    return df_limpio


df_cyber_security_clean = limpiar_cyber_security(df_cyber_security)

df_cyber_security_clean.head()

,country,region,cei,gci,ncsi,ddl,has_missing_security_index
0,Afghanistan,Asia-Pacific,1.000,5.20,11.69,19.50,False
1,Albania,Europe,0.566,64.32,62.34,48.74,False
2,Algeria,Africa,0.721,33.95,33.77,42.81,False
3,Andorra,Europe,NaN,26.38,NaN,NaN,True
4,Angola,Africa,NaN,12.99,9.09,22.69,True


### 3.3 Limpieza del dataset Internet Users by Country

En esta fuente se limpian nombres de columnas, textos y duplicados. La columna **Country or area** se renombra a **country** para facilitar la relación con las otras fuentes. También se convierten a tipo numérico las columnas de usuarios de internet, población, año y porcentaje de usuarios de internet.

Adicionalmente, se crea una columna auxiliar para identificar registros con datos digitales incompletos.

In [144]:
def limpiar_internet_users(df):
    """
    Limpia el DataFrame de usuarios de internet por país.
    """
    df_limpio = df.copy()
    
    # Corrección de nombres de columnas
    df_limpio.columns = [normalizar_nombre_columna(col) for col in df_limpio.columns]
    
    # Renombrar campo de país para homologarlo con las otras fuentes
    if "country_or_area" in df_limpio.columns:
        df_limpio = df_limpio.rename(columns={"country_or_area": "country"})
    
    # Eliminación de duplicados
    df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
    
    # Normalización de textos
    df_limpio = limpiar_columnas_texto(df_limpio)
    
    # Homologación de país
    df_limpio["country"] = df_limpio["country"].apply(homologar_pais)
    
    # Conversión de columnas numéricas
    columnas_numericas = [
        "internet_users",
        "population_2021",
        "year",
        "percentage_internet_users"
    ]
    df_limpio = convertir_columnas_numericas(df_limpio, columnas_numericas)
    
    # Conversión del año a entero nullable
    df_limpio["year"] = df_limpio["year"].astype("Int64")
    
    # Columna auxiliar para identificar registros incompletos en variables digitales clave
    df_limpio["has_missing_digital_data"] = df_limpio[columnas_numericas].isnull().any(axis=1)
    
    return df_limpio


df_internet_users_clean = limpiar_internet_users(df_internet_users)

df_internet_users_clean.head()

,country,subregion,region,internet_users,population_2021,year,percentage_internet_users,has_missing_digital_data
0,China,Eastern Asia,Asia,1102140000,1425893465,2022,77.3,False
1,India,Southern Asia,Asia,881250000,1407563842,2024,62.6,False
2,United States,Northern America,Americas,311300000,336997624,2023,92.4,False
3,Indonesia,South-eastern Asia,Asia,215626156,273753191,2023,78.8,False
4,Pakistan,Southern Asia,Asia,170000000,240000000,2022,70.8,False


### 3.4 Validación de la limpieza aplicada

Después de limpiar cada fuente, se realiza una validación general para comparar la cantidad de filas, columnas, duplicados y valores nulos antes y después del proceso. Esta revisión permite confirmar que la limpieza no eliminó información de forma innecesaria y que las transformaciones aplicadas mantienen la estructura esperada de los datos.

In [145]:
# Consolidar los DataFrames limpios en un diccionario
clean_datasets = {
    "Global Cybersecurity Threats": df_threats_clean,
    "Cyber Security Index": df_cyber_security_clean,
    "Internet Users by Country": df_internet_users_clean
}

# Resumen comparativo antes y después de la limpieza
resumenes_limpieza = [
    resumen_limpieza(nombre, datasets[nombre], clean_datasets[nombre])
    for nombre in datasets.keys()
]

df_resumen_limpieza = pd.DataFrame(resumenes_limpieza)
df_resumen_limpieza

,fuente,filas_originales,filas_limpias,columnas_originales,columnas_limpias,duplicados_originales,duplicados_limpios,nulos_originales,nulos_limpios
0,Global Cybersecurity Threats,3000,3000,10,10,0,0,0,0
1,Cyber Security Index,192,192,6,7,0,0,151,151
2,Internet Users by Country,215,215,7,8,0,0,0,0


In [146]:
# Revisión de tipos de datos después de la limpieza
for nombre, df in clean_datasets.items():
    print(f"\nTipos de datos después de la limpieza: {nombre}")
    display(pd.DataFrame({
        "columna": df.columns,
        "tipo_dato": df.dtypes.astype(str).values,
        "valores_nulos": df.isnull().sum().values,
        "valores_unicos": df.nunique().values
    }))


Tipos de datos después de la limpieza: Global Cybersecurity Threats


,columna,tipo_dato,valores_nulos,valores_unicos
0,country,str,0,10
1,year,Int64,0,10
2,attack_type,str,0,6
3,target_industry,str,0,7
4,financial_loss_in_million,float64,0,2536
5,number_of_affected_users,int64,0,2998
6,attack_source,str,0,4
7,security_vulnerability_type,str,0,4
8,defense_mechanism_used,str,0,5
9,incident_resolution_time_in_hours,int64,0,72



Tipos de datos después de la limpieza: Cyber Security Index


,columna,tipo_dato,valores_nulos,valores_unicos
0,country,str,0,192
1,region,str,0,5
2,cei,float64,84,85
3,gci,float64,2,180
4,ncsi,float64,25,69
5,ddl,float64,40,150
6,has_missing_security_index,bool,0,2



Tipos de datos después de la limpieza: Internet Users by Country


,columna,tipo_dato,valores_nulos,valores_unicos
0,country,str,0,215
1,subregion,str,0,22
2,region,str,0,5
3,internet_users,int64,0,215
4,population_2021,int64,0,215
5,year,Int64,0,5
6,percentage_internet_users,float64,0,195
7,has_missing_digital_data,bool,0,1


In [147]:
# Validación de países después de la homologación
paises_threats_clean = set(df_threats_clean["country"].dropna().unique())
paises_cyber_security_clean = set(df_cyber_security_clean["country"].dropna().unique())
paises_internet_users_clean = set(df_internet_users_clean["country"].dropna().unique())

paises_en_todas_las_fuentes = (
    paises_threats_clean
    .intersection(paises_cyber_security_clean)
    .intersection(paises_internet_users_clean)
)

print("Países en Global Cybersecurity Threats:", len(paises_threats_clean))
print("Países en Cyber Security Index:", len(paises_cyber_security_clean))
print("Países en Internet Users by Country:", len(paises_internet_users_clean))
print("\nPaíses del dataset de amenazas que también existen en las tres fuentes:", len(paises_en_todas_las_fuentes))
print(sorted(paises_en_todas_las_fuentes))

Países en Global Cybersecurity Threats: 10
Países en Cyber Security Index: 192
Países en Internet Users by Country: 215

Países del dataset de amenazas que también existen en las tres fuentes: 10
['Australia', 'Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'Russia', 'United Kingdom', 'United States']


### Hallazgos de la limpieza de datos

En la limpieza del dataset **Global Cybersecurity Threats** se corrigieron los nombres de columnas a formato `snake_case`, se eliminaron posibles duplicados, se normalizaron los textos y se homologaron nombres de países como **USA** y **UK**. Además, se validaron las columnas numéricas relacionadas con año, pérdidas financieras, usuarios afectados y tiempo de resolución.

En el dataset **Cyber Security Index** se aplicó la misma normalización de columnas y textos. También se corrigió la categoría regional **Asia-Pasific** a **Asia-Pacific**. Los indicadores CEI, GCI, NCSI y DDL fueron convertidos a tipo numérico y se creó la columna `has_missing_security_index` para identificar registros con indicadores incompletos.

En el dataset **Internet Users by Country** se renombró la columna `country_or_area` a `country`, permitiendo que el campo de país tenga el mismo nombre que en las demás fuentes. También se normalizaron textos, se convirtieron variables numéricas y se creó la columna `has_missing_digital_data` para marcar registros con información digital incompleta.

Esta fase permite contar con tres DataFrames limpios y más consistentes: `df_threats_clean`, `df_cyber_security_clean` y `df_internet_users_clean`. Estos serán utilizados en las siguientes etapas para seleccionar columnas relevantes, crear nuevas variables, generar identificadores y validar los resultados transformados.

## 4. Selección de columnas relevantes

En esta fase se evalúan las columnas disponibles después del proceso de limpieza para determinar cuáles serán utilizadas en el proyecto. La selección de columnas permite conservar únicamente la información necesaria para el análisis, reducir ruido en los datos y facilitar la integración entre las fuentes.

Para este proyecto, se mantienen columnas relacionadas con identificación geográfica, periodo de análisis, características de los incidentes, impacto económico, usuarios afectados, indicadores de ciberseguridad y nivel de conectividad digital.

También se identifican las columnas que pueden servir como campos de relación entre fuentes, principalmente **country** y, en ciertos análisis, **year** y **region**.

### 4.1 Definición de columnas a mantener

A continuación se definen las columnas que se conservarán en cada DataFrame limpio. Estas columnas fueron seleccionadas porque aportan valor al objetivo del proyecto y permiten analizar los incidentes de ciberseguridad junto con indicadores digitales y de madurez en seguridad por país.

In [148]:
# Columnas seleccionadas para el dataset de amenazas globales
columnas_threats = [
    "country",
    "year",
    "attack_type",
    "target_industry",
    "financial_loss_in_million",
    "number_of_affected_users",
    "attack_source",
    "security_vulnerability_type",
    "defense_mechanism_used",
    "incident_resolution_time_in_hours"
]

# Columnas seleccionadas para el dataset de índices de ciberseguridad
columnas_cyber_security = [
    "country",
    "region",
    "cei",
    "gci",
    "ncsi",
    "ddl",
    "has_missing_security_index"
]

# Columnas seleccionadas para el dataset de usuarios de internet
columnas_internet_users = [
    "country",
    "subregion",
    "region",
    "internet_users",
    "population_2021",
    "year",
    "percentage_internet_users",
    "has_missing_digital_data"
]

# Crear DataFrames con las columnas seleccionadas
df_threats_selected = df_threats_clean[columnas_threats].copy()
df_cyber_security_selected = df_cyber_security_clean[columnas_cyber_security].copy()
df_internet_users_selected = df_internet_users_clean[columnas_internet_users].copy()

print("Columnas relevantes seleccionadas correctamente.")

Columnas relevantes seleccionadas correctamente.


### 4.2 Revisión de columnas mantenidas y eliminadas

En esta sección se compara la estructura de los DataFrames limpios frente a los DataFrames seleccionados. Esto permite identificar qué columnas se mantienen y cuáles se eliminan en esta etapa.

En este caso, la mayoría de columnas se conservan porque las fuentes originales ya contienen variables relevantes para el análisis del proyecto. Sin embargo, se deja el proceso documentado para evidenciar la evaluación realizada.

In [149]:
def obtener_columnas_eliminadas(df_original, columnas_mantenidas):
    """
    Retorna las columnas que no fueron seleccionadas para el DataFrame final.
    """
    return [col for col in df_original.columns if col not in columnas_mantenidas]


seleccion_columnas = pd.DataFrame({
    "fuente": [
        "Global Cybersecurity Threats",
        "Cyber Security Index",
        "Internet Users by Country"
    ],
    "columnas_mantenidas": [
        columnas_threats,
        columnas_cyber_security,
        columnas_internet_users
    ],
    "columnas_eliminadas": [
        obtener_columnas_eliminadas(df_threats_clean, columnas_threats),
        obtener_columnas_eliminadas(df_cyber_security_clean, columnas_cyber_security),
        obtener_columnas_eliminadas(df_internet_users_clean, columnas_internet_users)
    ]
})

seleccion_columnas

,fuente,columnas_mantenidas,columnas_eliminadas
0,Global Cybersecurity Threats,"[country, year, attack_type, target_industry, ...",[]
1,Cyber Security Index,"[country, region, cei, gci, ncsi, ddl, has_mis...",[]
2,Internet Users by Country,"[country, subregion, region, internet_users, p...",[]


### 4.3 Justificación de columnas seleccionadas

Cada columna seleccionada cumple una función específica dentro del proyecto. Algunas columnas permiten identificar el país o periodo del registro, mientras que otras permiten analizar el tipo de ataque, el impacto del incidente, el nivel de madurez en ciberseguridad o la conectividad digital del país.

In [150]:
justificacion_columnas = pd.DataFrame([
    # Global Cybersecurity Threats
    {
        "fuente": "Global Cybersecurity Threats",
        "columna": "country",
        "decision": "Se mantiene",
        "justificacion": "Identifica el país donde se registra el incidente y permite relacionar esta fuente con las demás.",
        "uso_relacion": "Sí"
    },
    {
        "fuente": "Global Cybersecurity Threats",
        "columna": "year",
        "decision": "Se mantiene",
        "justificacion": "Permite analizar la evolución temporal de los incidentes de ciberseguridad.",
        "uso_relacion": "Parcial"
    },
    {
        "fuente": "Global Cybersecurity Threats",
        "columna": "attack_type",
        "decision": "Se mantiene",
        "justificacion": "Permite clasificar los incidentes según el tipo de ataque.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Global Cybersecurity Threats",
        "columna": "target_industry",
        "decision": "Se mantiene",
        "justificacion": "Permite identificar las industrias más afectadas por incidentes.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Global Cybersecurity Threats",
        "columna": "financial_loss_in_million",
        "decision": "Se mantiene",
        "justificacion": "Mide el impacto económico del incidente.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Global Cybersecurity Threats",
        "columna": "number_of_affected_users",
        "decision": "Se mantiene",
        "justificacion": "Permite evaluar el alcance del incidente en cantidad de usuarios afectados.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Global Cybersecurity Threats",
        "columna": "attack_source",
        "decision": "Se mantiene",
        "justificacion": "Permite analizar el origen o fuente del ataque.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Global Cybersecurity Threats",
        "columna": "security_vulnerability_type",
        "decision": "Se mantiene",
        "justificacion": "Permite identificar vulnerabilidades comunes asociadas a los incidentes.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Global Cybersecurity Threats",
        "columna": "defense_mechanism_used",
        "decision": "Se mantiene",
        "justificacion": "Permite conocer los mecanismos de defensa aplicados frente a los ataques.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Global Cybersecurity Threats",
        "columna": "incident_resolution_time_in_hours",
        "decision": "Se mantiene",
        "justificacion": "Permite analizar el tiempo requerido para resolver los incidentes.",
        "uso_relacion": "No"
    },

    # Cyber Security Index
    {
        "fuente": "Cyber Security Index",
        "columna": "country",
        "decision": "Se mantiene",
        "justificacion": "Identifica el país evaluado y permite relacionar esta fuente con las demás.",
        "uso_relacion": "Sí"
    },
    {
        "fuente": "Cyber Security Index",
        "columna": "region",
        "decision": "Se mantiene",
        "justificacion": "Permite realizar análisis agrupados por región.",
        "uso_relacion": "Parcial"
    },
    {
        "fuente": "Cyber Security Index",
        "columna": "cei",
        "decision": "Se mantiene",
        "justificacion": "Indicador relacionado con exposición en ciberseguridad.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Cyber Security Index",
        "columna": "gci",
        "decision": "Se mantiene",
        "justificacion": "Indicador global de ciberseguridad útil para comparar países.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Cyber Security Index",
        "columna": "ncsi",
        "decision": "Se mantiene",
        "justificacion": "Indicador nacional de ciberseguridad que complementa el análisis de madurez.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Cyber Security Index",
        "columna": "ddl",
        "decision": "Se mantiene",
        "justificacion": "Permite analizar el nivel de desarrollo digital del país.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Cyber Security Index",
        "columna": "has_missing_security_index",
        "decision": "Se mantiene",
        "justificacion": "Permite identificar registros con indicadores de ciberseguridad incompletos.",
        "uso_relacion": "No"
    },

    # Internet Users by Country
    {
        "fuente": "Internet Users by Country",
        "columna": "country",
        "decision": "Se mantiene",
        "justificacion": "Identifica el país y permite relacionar esta fuente con las demás.",
        "uso_relacion": "Sí"
    },
    {
        "fuente": "Internet Users by Country",
        "columna": "subregion",
        "decision": "Se mantiene",
        "justificacion": "Permite un análisis geográfico más detallado que la región general.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Internet Users by Country",
        "columna": "region",
        "decision": "Se mantiene",
        "justificacion": "Permite análisis agrupados por región geográfica.",
        "uso_relacion": "Parcial"
    },
    {
        "fuente": "Internet Users by Country",
        "columna": "internet_users",
        "decision": "Se mantiene",
        "justificacion": "Permite medir el volumen de usuarios de internet por país.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Internet Users by Country",
        "columna": "population_2021",
        "decision": "Se mantiene",
        "justificacion": "Permite contextualizar los usuarios de internet frente a la población total.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Internet Users by Country",
        "columna": "year",
        "decision": "Se mantiene",
        "justificacion": "Permite identificar el año asociado al dato de usuarios de internet.",
        "uso_relacion": "Parcial"
    },
    {
        "fuente": "Internet Users by Country",
        "columna": "percentage_internet_users",
        "decision": "Se mantiene",
        "justificacion": "Permite analizar el nivel de penetración de internet por país.",
        "uso_relacion": "No"
    },
    {
        "fuente": "Internet Users by Country",
        "columna": "has_missing_digital_data",
        "decision": "Se mantiene",
        "justificacion": "Permite identificar registros con información digital incompleta.",
        "uso_relacion": "No"
    }
])

justificacion_columnas

,fuente,columna,decision,justificacion,uso_relacion
0,Global Cybersecurity Threats,country,Se mantiene,Identifica el país donde se registra el incide...,Sí
1,Global Cybersecurity Threats,year,Se mantiene,Permite analizar la evolución temporal de los ...,Parcial
2,Global Cybersecurity Threats,attack_type,Se mantiene,Permite clasificar los incidentes según el tip...,No
3,Global Cybersecurity Threats,target_industry,Se mantiene,Permite identificar las industrias más afectad...,No
4,Global Cybersecurity Threats,financial_loss_in_million,Se mantiene,Mide el impacto económico del incidente.,No
5,Global Cybersecurity Threats,number_of_affected_users,Se mantiene,Permite evaluar el alcance del incidente en ca...,No
6,Global Cybersecurity Threats,attack_source,Se mantiene,Permite analizar el origen o fuente del ataque.,No
7,Global Cybersecurity Threats,security_vulnerability_type,Se mantiene,Permite identificar vulnerabilidades comunes a...,No
8,Global Cybersecurity Threats,defense_mechanism_used,Se mantiene,Permite conocer los mecanismos de defensa apli...,No
9,Global Cybersecurity Threats,incident_resolution_time_in_hours,Se mantiene,Permite analizar el tiempo requerido para reso...,No


### 4.4 Columnas de relación entre fuentes

Para integrar o comparar las fuentes, se identifican las columnas que podrían actuar como campos de relación. La columna más importante es **country**, ya que está presente en las tres fuentes después de la limpieza. El campo **year** puede apoyar análisis temporales, pero debe usarse con cuidado porque no todas las fuentes tienen el mismo rango temporal. La columna **region** permite análisis agrupados, aunque las categorías regionales deben mantenerse normalizadas.

In [151]:
columnas_para_relacion = pd.DataFrame([
    {
        "columna_relacion": "country",
        "fuentes": "Global Cybersecurity Threats, Cyber Security Index, Internet Users by Country",
        "tipo_relacion": "Principal",
        "descripcion": "Permite relacionar los incidentes, indicadores de ciberseguridad y usuarios de internet por país."
    },
    {
        "columna_relacion": "year",
        "fuentes": "Global Cybersecurity Threats, Internet Users by Country",
        "tipo_relacion": "Parcial / temporal",
        "descripcion": "Permite análisis por año, aunque debe validarse porque el índice de ciberseguridad no tiene columna de año."
    },
    {
        "columna_relacion": "region",
        "fuentes": "Cyber Security Index, Internet Users by Country",
        "tipo_relacion": "Agrupación geográfica",
        "descripcion": "Permite analizar resultados por región, siempre que las categorías estén normalizadas."
    }
])

columnas_para_relacion

,columna_relacion,fuentes,tipo_relacion,descripcion
0,country,"Global Cybersecurity Threats, Cyber Security I...",Principal,"Permite relacionar los incidentes, indicadores..."
1,year,"Global Cybersecurity Threats, Internet Users b...",Parcial / temporal,"Permite análisis por año, aunque debe validars..."
2,region,"Cyber Security Index, Internet Users by Country",Agrupación geográfica,"Permite analizar resultados por región, siempr..."


### 4.5 Vista previa de los DataFrames seleccionados

Finalmente, se muestran las primeras filas de cada DataFrame seleccionado para validar que las columnas conservadas sean las esperadas y que la estructura sea adecuada para continuar con las siguientes transformaciones.

In [152]:
print("DataFrame seleccionado: Global Cybersecurity Threats")
display(df_threats_selected.head())

print("DataFrame seleccionado: Cyber Security Index")
display(df_cyber_security_selected.head())

print("DataFrame seleccionado: Internet Users by Country")
display(df_internet_users_selected.head())

DataFrame seleccionado: Global Cybersecurity Threats


,country,year,attack_type,target_industry,financial_loss_in_million,number_of_affected_users,attack_source,security_vulnerability_type,defense_mechanism_used,incident_resolution_time_in_hours
0,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63
1,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71
2,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20
3,United Kingdom,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7
4,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68


DataFrame seleccionado: Cyber Security Index


,country,region,cei,gci,ncsi,ddl,has_missing_security_index
0,Afghanistan,Asia-Pacific,1.000,5.20,11.69,19.50,False
1,Albania,Europe,0.566,64.32,62.34,48.74,False
2,Algeria,Africa,0.721,33.95,33.77,42.81,False
3,Andorra,Europe,NaN,26.38,NaN,NaN,True
4,Angola,Africa,NaN,12.99,9.09,22.69,True


DataFrame seleccionado: Internet Users by Country


,country,subregion,region,internet_users,population_2021,year,percentage_internet_users,has_missing_digital_data
0,China,Eastern Asia,Asia,1102140000,1425893465,2022,77.3,False
1,India,Southern Asia,Asia,881250000,1407563842,2024,62.6,False
2,United States,Northern America,Americas,311300000,336997624,2023,92.4,False
3,Indonesia,South-eastern Asia,Asia,215626156,273753191,2023,78.8,False
4,Pakistan,Southern Asia,Asia,170000000,240000000,2022,70.8,False


### Hallazgos de la selección de columnas relevantes

Después de evaluar las columnas disponibles en las tres fuentes limpias, se decidió conservar las variables que aportan valor directo al análisis del proyecto. En el dataset de amenazas se mantienen columnas relacionadas con país, año, tipo de ataque, industria afectada, pérdidas financieras, usuarios afectados, fuente del ataque, vulnerabilidad, mecanismo de defensa y tiempo de resolución.

En el dataset de índices de ciberseguridad se mantienen el país, la región y los indicadores CEI, GCI, NCSI y DDL, además de la columna auxiliar que identifica registros con datos incompletos. Estas columnas permiten complementar el análisis de incidentes con indicadores de madurez, exposición y desarrollo digital.

En el dataset de usuarios de internet se conservan país, subregión, región, usuarios de internet, población, año, porcentaje de usuarios de internet y la columna auxiliar de datos incompletos. Estas variables permiten contextualizar el nivel de conectividad digital de cada país.

La columna principal para relacionar las tres fuentes será **country**, debido a que está presente en todos los DataFrames después del proceso de limpieza. La columna **year** puede utilizarse para análisis temporales entre amenazas y usuarios de internet, mientras que **region** puede apoyar análisis geográficos entre índices de ciberseguridad y usuarios de internet.

En esta etapa no se eliminan columnas críticas, ya que las fuentes contienen variables directamente relacionadas con el objetivo del proyecto. Sin embargo, la selección queda documentada y se crean tres DataFrames específicos para continuar el proceso: `df_threats_selected`, `df_cyber_security_selected` y `df_internet_users_selected`.

## 5. Transformaciones requeridas

En esta fase se aplican transformaciones adicionales sobre los DataFrames seleccionados. El objetivo es enriquecer los datos mediante la creación de nuevas columnas, categorización de variables, conversión de formatos y generación de campos derivados que faciliten el análisis posterior.

Las transformaciones aplicadas se enfocan en el dominio del proyecto: ciberseguridad global e indicadores digitales por país. Por esta razón, se crean categorías para analizar el impacto económico de los incidentes, la cantidad de usuarios afectados, el tiempo de resolución, el nivel de madurez en ciberseguridad y el nivel de penetración de internet.

También se generan claves limpias de país y año para facilitar futuras relaciones entre las fuentes.

### 5.1 Funciones auxiliares para transformación

Se crean funciones reutilizables para categorizar variables numéricas y generar claves limpias. Esto permite mantener el código organizado, evitar repetición y documentar de forma clara la lógica de transformación aplicada.

In [153]:
def crear_clave_texto(valor):
    """
    Crea una clave limpia a partir de un texto.
    Ejemplo: 'United States' -> 'united_states'
    """
    if pd.isna(valor):
        return pd.NA
    
    valor = str(valor).strip().lower()
    valor = unicodedata.normalize("NFKD", valor).encode("ascii", "ignore").decode("utf-8")
    valor = re.sub(r"[^a-z0-9]+", "_", valor)
    valor = re.sub(r"_+", "_", valor)
    valor = valor.strip("_")
    
    return valor


def categorizar_perdida_financiera(valor):
    """
    Clasifica la pérdida financiera del incidente en millones de dólares.
    """
    if pd.isna(valor):
        return "Sin dato"
    if valor < 25:
        return "Baja"
    if valor < 50:
        return "Media"
    if valor < 75:
        return "Alta"
    return "Crítica"


def categorizar_usuarios_afectados(valor):
    """
    Clasifica la cantidad de usuarios afectados por un incidente.
    """
    if pd.isna(valor):
        return "Sin dato"
    if valor < 250_000:
        return "Bajo impacto"
    if valor < 500_000:
        return "Impacto medio"
    if valor < 750_000:
        return "Impacto alto"
    return "Impacto crítico"


def categorizar_tiempo_resolucion(valor):
    """
    Clasifica el tiempo de resolución del incidente en horas.
    """
    if pd.isna(valor):
        return "Sin dato"
    if valor <= 24:
        return "Resolución rápida"
    if valor <= 48:
        return "Resolución media"
    return "Resolución lenta"


def categorizar_indicador_0_100(valor):
    """
    Clasifica indicadores expresados en escala de 0 a 100.
    """
    if pd.isna(valor):
        return "Sin dato"
    if valor < 40:
        return "Bajo"
    if valor < 70:
        return "Medio"
    if valor < 85:
        return "Alto"
    return "Muy alto"


def categorizar_penetracion_internet(valor):
    """
    Clasifica el porcentaje de usuarios de internet por país.
    """
    if pd.isna(valor):
        return "Sin dato"
    if valor < 40:
        return "Baja"
    if valor < 70:
        return "Media"
    if valor < 90:
        return "Alta"
    return "Muy alta"

print("Funciones auxiliares de transformación creadas correctamente.")

Funciones auxiliares de transformación creadas correctamente.


#### Explicación de las funciones auxiliares

Las funciones creadas en este bloque permiten reutilizar reglas de transformación en diferentes DataFrames. La función `crear_clave_texto` genera claves limpias en minúsculas y sin caracteres especiales, útiles para relacionar datos. Las funciones de categorización convierten valores numéricos en categorías descriptivas, lo cual facilita la interpretación de los resultados en análisis o reportes.

### 5.2 Transformaciones del dataset Global Cybersecurity Threats

En el dataset de amenazas se crean nuevas columnas para clasificar la pérdida financiera, el impacto por usuarios afectados y el tiempo de resolución. También se convierte la pérdida financiera de millones de dólares a dólares y se genera una clave limpia de país y una clave país-año.

Estas transformaciones permiten analizar los incidentes de una forma más interpretativa y facilitan futuras relaciones con otras fuentes.

In [154]:
df_threats_transformed = df_threats_selected.copy()

# Clave limpia de país para facilitar relaciones y validaciones
df_threats_transformed["country_key"] = df_threats_transformed["country"].apply(lambda x: crear_clave_texto(x))

# Clave compuesta país-año para posibles análisis temporales
df_threats_transformed["country_year_key"] = (
    df_threats_transformed["country_key"]
    + "_"
    + df_threats_transformed["year"].astype(str)
)

# Conversión de pérdida financiera de millones a dólares
df_threats_transformed["financial_loss_usd"] = (
    df_threats_transformed["financial_loss_in_million"] * 1_000_000
)

# Conversión de año a una fecha de referencia para análisis temporales
df_threats_transformed["incident_year_start_date"] = pd.to_datetime(
    df_threats_transformed["year"].astype(str) + "-01-01",
    errors="coerce"
)

# Categorización de variables usando funciones y lambda
df_threats_transformed["financial_loss_category"] = df_threats_transformed["financial_loss_in_million"].apply(
    lambda valor: categorizar_perdida_financiera(valor)
)

df_threats_transformed["affected_users_category"] = df_threats_transformed["number_of_affected_users"].apply(
    lambda valor: categorizar_usuarios_afectados(valor)
)

df_threats_transformed["resolution_time_category"] = df_threats_transformed["incident_resolution_time_in_hours"].apply(
    lambda valor: categorizar_tiempo_resolucion(valor)
)

df_threats_transformed.head()

,country,year,attack_type,target_industry,financial_loss_in_million,number_of_affected_users,attack_source,security_vulnerability_type,defense_mechanism_used,incident_resolution_time_in_hours,country_key,country_year_key,financial_loss_usd,incident_year_start_date,financial_loss_category,affected_users_category,resolution_time_category
0,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63,china,china_2019,80530000.0,2019-01-01,Crítica,Impacto crítico,Resolución lenta
1,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71,china,china_2019,62190000.0,2019-01-01,Alta,Impacto medio,Resolución lenta
2,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20,india,india_2017,38650000.0,2017-01-01,Media,Impacto alto,Resolución rápida
3,United Kingdom,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7,united_kingdom,united_kingdom_2024,41440000.0,2024-01-01,Media,Impacto alto,Resolución rápida
4,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68,germany,germany_2018,74410000.0,2018-01-01,Alta,Impacto crítico,Resolución lenta


#### Explicación de las transformaciones aplicadas a Global Cybersecurity Threats

En este bloque se expande el DataFrame de amenazas con columnas derivadas. La columna `country_key` normaliza el país para futuras relaciones, mientras que `country_year_key` permite análisis combinando país y año. También se convierte la pérdida financiera de millones de dólares a dólares mediante `financial_loss_usd`. Finalmente, se crean categorías para clasificar la pérdida financiera, el impacto por usuarios afectados y el tiempo de resolución del incidente.

### 5.3 Transformaciones del dataset Cyber Security Index

En el dataset de indicadores de ciberseguridad se generan columnas derivadas para categorizar los indicadores GCI, NCSI y DDL. También se transforma CEI a una escala porcentual y se crea una columna que indica cuántos indicadores están disponibles por registro.

Estas transformaciones permiten interpretar más fácilmente el nivel de madurez, exposición y desarrollo digital de cada país.

In [155]:
df_cyber_security_transformed = df_cyber_security_selected.copy()

# Clave limpia de país
df_cyber_security_transformed["country_key"] = df_cyber_security_transformed["country"].apply(
    lambda x: crear_clave_texto(x)
)

# Conversión de CEI a escala porcentual para facilitar su lectura
df_cyber_security_transformed["cei_percentage"] = df_cyber_security_transformed["cei"] * 100

# Cantidad de indicadores disponibles por país
indicadores_ciberseguridad = ["cei", "gci", "ncsi", "ddl"]
df_cyber_security_transformed["available_security_indicators"] = df_cyber_security_transformed[
    indicadores_ciberseguridad
].notnull().sum(axis=1)

# Categorización de indicadores principales
df_cyber_security_transformed["gci_level"] = df_cyber_security_transformed["gci"].apply(
    lambda valor: categorizar_indicador_0_100(valor)
)

df_cyber_security_transformed["ncsi_level"] = df_cyber_security_transformed["ncsi"].apply(
    lambda valor: categorizar_indicador_0_100(valor)
)

df_cyber_security_transformed["ddl_level"] = df_cyber_security_transformed["ddl"].apply(
    lambda valor: categorizar_indicador_0_100(valor)
)

# Indicador inverso para facilitar lectura de completitud
df_cyber_security_transformed["has_complete_security_index"] = ~df_cyber_security_transformed[
    "has_missing_security_index"
]

df_cyber_security_transformed.head()

,country,region,cei,gci,ncsi,ddl,has_missing_security_index,country_key,cei_percentage,available_security_indicators,gci_level,ncsi_level,ddl_level,has_complete_security_index
0,Afghanistan,Asia-Pacific,1.000,5.20,11.69,19.50,False,afghanistan,100.0,4,Bajo,Bajo,Bajo,True
1,Albania,Europe,0.566,64.32,62.34,48.74,False,albania,56.6,4,Medio,Medio,Medio,True
2,Algeria,Africa,0.721,33.95,33.77,42.81,False,algeria,72.1,4,Bajo,Bajo,Medio,True
3,Andorra,Europe,NaN,26.38,NaN,NaN,True,andorra,NaN,1,Bajo,Sin dato,Sin dato,False
4,Angola,Africa,NaN,12.99,9.09,22.69,True,angola,NaN,3,Bajo,Bajo,Bajo,False


#### Explicación de las transformaciones aplicadas a Cyber Security Index

En este bloque se agregan columnas que facilitan la interpretación de los indicadores de ciberseguridad. `country_key` permite relacionar países de forma más estable. `cei_percentage` expresa CEI en escala porcentual. `available_security_indicators` cuenta cuántos indicadores están disponibles por país. Además, se categorizan GCI, NCSI y DDL en niveles descriptivos como bajo, medio, alto o muy alto.

### 5.4 Transformaciones del dataset Internet Users by Country

En el dataset de usuarios de internet se crean columnas para expresar usuarios y población en millones, convertir el porcentaje de usuarios de internet a proporción decimal y categorizar el nivel de penetración de internet.

También se genera una clave limpia de país y una clave país-año para facilitar futuras relaciones o análisis temporales.

In [156]:
df_internet_users_transformed = df_internet_users_selected.copy()

# Clave limpia de país
df_internet_users_transformed["country_key"] = df_internet_users_transformed["country"].apply(
    lambda x: crear_clave_texto(x)
)

# Clave país-año
df_internet_users_transformed["country_year_key"] = (
    df_internet_users_transformed["country_key"]
    + "_"
    + df_internet_users_transformed["year"].astype(str)
)

# Conversión de variables a millones para facilitar lectura
df_internet_users_transformed["internet_users_millions"] = (
    df_internet_users_transformed["internet_users"] / 1_000_000
).round(2)

df_internet_users_transformed["population_millions"] = (
    df_internet_users_transformed["population_2021"] / 1_000_000
).round(2)

# Conversión de porcentaje a proporción decimal
df_internet_users_transformed["internet_users_ratio"] = (
    df_internet_users_transformed["percentage_internet_users"] / 100
).round(4)

# Conversión de año a fecha de referencia
df_internet_users_transformed["internet_year_start_date"] = pd.to_datetime(
    df_internet_users_transformed["year"].astype(str) + "-01-01",
    errors="coerce"
)

# Categorización del nivel de penetración de internet
df_internet_users_transformed["internet_penetration_level"] = df_internet_users_transformed[
    "percentage_internet_users"
].apply(lambda valor: categorizar_penetracion_internet(valor))

df_internet_users_transformed.head()

,country,subregion,region,internet_users,population_2021,year,percentage_internet_users,has_missing_digital_data,country_key,country_year_key,internet_users_millions,population_millions,internet_users_ratio,internet_year_start_date,internet_penetration_level
0,China,Eastern Asia,Asia,1102140000,1425893465,2022,77.3,False,china,china_2022,1102.14,1425.89,0.773,2022-01-01,Alta
1,India,Southern Asia,Asia,881250000,1407563842,2024,62.6,False,india,india_2024,881.25,1407.56,0.626,2024-01-01,Media
2,United States,Northern America,Americas,311300000,336997624,2023,92.4,False,united_states,united_states_2023,311.30,337.00,0.924,2023-01-01,Muy alta
3,Indonesia,South-eastern Asia,Asia,215626156,273753191,2023,78.8,False,indonesia,indonesia_2023,215.63,273.75,0.788,2023-01-01,Alta
4,Pakistan,Southern Asia,Asia,170000000,240000000,2022,70.8,False,pakistan,pakistan_2022,170.00,240.00,0.708,2022-01-01,Alta


#### Explicación de las transformaciones aplicadas a Internet Users by Country

En este bloque se crean columnas derivadas para mejorar la lectura de los datos digitales. `internet_users_millions` y `population_millions` expresan cantidades grandes en millones. `internet_users_ratio` convierte el porcentaje de usuarios de internet en proporción decimal. `internet_penetration_level` clasifica el nivel de penetración de internet y `country_year_key` permite futuros análisis por país y año.

### 5.5 Expansión e integración de datos derivados

Como parte de la expansión del análisis, se crea una vista integrada a partir del dataset de amenazas. Esta vista incorpora información de los indicadores de ciberseguridad y usuarios de internet utilizando el país como columna de relación.

Esta integración permite analizar cada incidente considerando también el contexto digital y de ciberseguridad del país asociado.

In [157]:
# Preparar columnas de Cyber Security para la integración por país
cyber_security_context = df_cyber_security_transformed[[
    "country",
    "region",
    "cei",
    "gci",
    "ncsi",
    "ddl",
    "cei_percentage",
    "gci_level",
    "ncsi_level",
    "ddl_level",
    "available_security_indicators",
    "has_complete_security_index"
]].rename(columns={
    "region": "security_region"
})

# Preparar columnas de Internet Users para la integración por país
internet_users_context = df_internet_users_transformed[[
    "country",
    "subregion",
    "region",
    "year",
    "internet_users",
    "population_2021",
    "percentage_internet_users",
    "internet_users_millions",
    "population_millions",
    "internet_users_ratio",
    "internet_penetration_level"
]].rename(columns={
    "region": "internet_region",
    "year": "internet_data_year"
})

# Integrar contexto de ciberseguridad e internet al dataset de amenazas.
# Se usa left join para conservar todos los incidentes de ciberseguridad.
df_threats_enriched = (
    df_threats_transformed
    .merge(cyber_security_context, on="country", how="left")
    .merge(internet_users_context, on="country", how="left")
)

df_threats_enriched.head()

,country,year,attack_type,target_industry,financial_loss_in_million,number_of_affected_users,attack_source,security_vulnerability_type,defense_mechanism_used,incident_resolution_time_in_hours,country_key,country_year_key,financial_loss_usd,incident_year_start_date,financial_loss_category,affected_users_category,resolution_time_category,security_region,cei,gci,ncsi,ddl,cei_percentage,gci_level,ncsi_level,ddl_level,available_security_indicators,has_complete_security_index,subregion,internet_region,internet_data_year,internet_users,population_2021,percentage_internet_users,internet_users_millions,population_millions,internet_users_ratio,internet_penetration_level
0,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63,china,china_2019,80530000.0,2019-01-01,Crítica,Impacto crítico,Resolución lenta,Asia-Pacific,0.483,92.53,51.95,62.41,48.3,Muy alto,Medio,Medio,4,True,Eastern Asia,Asia,2022,1102140000,1425893465,77.3,1102.14,1425.89,0.773,Alta
1,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71,china,china_2019,62190000.0,2019-01-01,Alta,Impacto medio,Resolución lenta,Asia-Pacific,0.483,92.53,51.95,62.41,48.3,Muy alto,Medio,Medio,4,True,Eastern Asia,Asia,2022,1102140000,1425893465,77.3,1102.14,1425.89,0.773,Alta
2,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20,india,india_2017,38650000.0,2017-01-01,Media,Impacto alto,Resolución rápida,Asia-Pacific,0.597,97.50,59.74,40.02,59.7,Muy alto,Medio,Medio,4,True,Southern Asia,Asia,2024,881250000,1407563842,62.6,881.25,1407.56,0.626,Media
3,United Kingdom,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7,united_kingdom,united_kingdom_2024,41440000.0,2024-01-01,Media,Impacto alto,Resolución rápida,Europe,0.207,99.54,89.61,79.96,20.7,Muy alto,Muy alto,Alto,4,True,Northern Europe,Europe,2021,65001016,67281039,96.6,65.00,67.28,0.966,Muy alta
4,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68,germany,germany_2018,74410000.0,2018-01-01,Alta,Impacto crítico,Resolución lenta,Europe,0.241,97.41,90.91,80.01,24.1,Muy alto,Muy alto,Alto,4,True,Western Europe,Europe,2021,77794405,83408554,93.3,77.79,83.41,0.933,Muy alta


#### Explicación de la expansión e integración de datos

En este bloque se genera una vista integrada llamada `df_threats_enriched`. Esta vista toma como base los incidentes de ciberseguridad y agrega información contextual del índice de ciberseguridad y usuarios de internet por país. La integración se realiza mediante `country` y se utiliza `left join` para conservar todos los registros del dataset principal de amenazas.

### 5.6 Validación de transformaciones aplicadas

Después de crear las columnas derivadas y la vista integrada, se revisa la estructura de los DataFrames transformados. Esta validación permite confirmar que las nuevas columnas fueron creadas correctamente y que la integración por país conserva los registros del dataset principal de amenazas.

In [158]:
# Resumen de columnas creadas en cada DataFrame transformado
resumen_transformaciones = pd.DataFrame([
    {
        "dataframe": "df_threats_transformed",
        "filas": df_threats_transformed.shape[0],
        "columnas": df_threats_transformed.shape[1],
        "nuevas_columnas": [
            "country_key",
            "country_year_key",
            "financial_loss_usd",
            "incident_year_start_date",
            "financial_loss_category",
            "affected_users_category",
            "resolution_time_category"
        ]
    },
    {
        "dataframe": "df_cyber_security_transformed",
        "filas": df_cyber_security_transformed.shape[0],
        "columnas": df_cyber_security_transformed.shape[1],
        "nuevas_columnas": [
            "country_key",
            "cei_percentage",
            "available_security_indicators",
            "gci_level",
            "ncsi_level",
            "ddl_level",
            "has_complete_security_index"
        ]
    },
    {
        "dataframe": "df_internet_users_transformed",
        "filas": df_internet_users_transformed.shape[0],
        "columnas": df_internet_users_transformed.shape[1],
        "nuevas_columnas": [
            "country_key",
            "country_year_key",
            "internet_users_millions",
            "population_millions",
            "internet_users_ratio",
            "internet_year_start_date",
            "internet_penetration_level"
        ]
    },
    {
        "dataframe": "df_threats_enriched",
        "filas": df_threats_enriched.shape[0],
        "columnas": df_threats_enriched.shape[1],
        "nuevas_columnas": "Vista integrada con indicadores de ciberseguridad e internet"
    }
])

resumen_transformaciones

,dataframe,filas,columnas,nuevas_columnas
0,df_threats_transformed,3000,17,"[country_key, country_year_key, financial_loss..."
1,df_cyber_security_transformed,192,14,"[country_key, cei_percentage, available_securi..."
2,df_internet_users_transformed,215,15,"[country_key, country_year_key, internet_users..."
3,df_threats_enriched,3000,38,Vista integrada con indicadores de ciberseguri...


#### Explicación de la validación de transformaciones

En este bloque se resume la cantidad de filas, columnas y nuevas variables creadas en cada DataFrame transformado. Esta revisión permite comprobar que las transformaciones se aplicaron correctamente y facilita documentar qué columnas fueron derivadas durante la fase de transformación.

In [159]:
# Validar que la integración no haya eliminado incidentes del dataset principal
print("Filas en df_threats_transformed:", df_threats_transformed.shape[0])
print("Filas en df_threats_enriched:", df_threats_enriched.shape[0])

if df_threats_transformed.shape[0] == df_threats_enriched.shape[0]:
    print("La integración conserva todos los registros del dataset de amenazas.")
else:
    print("Revisar integración: cambió la cantidad de registros del dataset principal.")

Filas en df_threats_transformed: 3000
Filas en df_threats_enriched: 3000
La integración conserva todos los registros del dataset de amenazas.


#### Explicación de la validación de integración

En este bloque se compara la cantidad de filas del DataFrame de amenazas transformado con la vista integrada. El objetivo es confirmar que la integración no eliminó registros del dataset principal. Si ambas cantidades coinciden, significa que todos los incidentes se conservaron después del cruce con las fuentes de contexto.

### Hallazgos de las transformaciones requeridas

En esta fase se generaron nuevas columnas derivadas que enriquecen los datos limpios y seleccionados. En el dataset de amenazas se crearon claves de país, claves país-año, pérdida financiera convertida a dólares, fecha de referencia anual y categorías para pérdida financiera, usuarios afectados y tiempo de resolución.

En el dataset de índices de ciberseguridad se agregó una clave limpia de país, el CEI expresado en porcentaje, la cantidad de indicadores disponibles por país y categorías para interpretar los niveles de GCI, NCSI y DDL. También se creó una columna que permite identificar registros con indicadores completos.

En el dataset de usuarios de internet se generaron columnas para usuarios y población expresados en millones, porcentaje convertido a proporción decimal, fecha de referencia anual y nivel de penetración de internet.

Además, se creó una vista integrada llamada `df_threats_enriched`, que combina los incidentes de ciberseguridad con el contexto de indicadores de seguridad e internet por país. Esta vista conserva todos los registros del dataset principal de amenazas y permite realizar análisis más completos sobre el impacto de los incidentes según el contexto digital de cada país.

Estas transformaciones permiten que los datos sean más interpretables, comparables y útiles para etapas posteriores de análisis, validación o carga en una base de datos.

## 6. Generación de identificadores

En esta fase se generan identificadores numéricos únicos y secuenciales para facilitar el manejo de registros, la integración entre fuentes y una posible carga posterior hacia una base de datos relacional.

Aunque los DataFrames ya contienen campos como país o año, estos valores no siempre son ideales como identificadores técnicos. Por eso, se crean IDs numéricos que permiten organizar los datos de forma más estructurada y mantener relaciones más claras entre tablas.

### 6.1 Creación de una dimensión de países

Primero se crea una tabla auxiliar llamada `df_country_dimension`, que contiene todos los países presentes en las fuentes transformadas. A cada país se le asigna un identificador numérico secuencial llamado `country_id`.

Este identificador se utilizará para relacionar los DataFrames sin depender directamente del texto del país, reduciendo problemas por diferencias de escritura o formato.

In [160]:
# Consolidar países únicos desde las tres fuentes transformadas
paises_unicos = sorted(
    set(df_threats_transformed["country"].dropna())
    .union(set(df_cyber_security_transformed["country"].dropna()))
    .union(set(df_internet_users_transformed["country"].dropna()))
)

# Crear dimensión de países con identificador numérico secuencial
df_country_dimension = pd.DataFrame({
    "country_id": range(1, len(paises_unicos) + 1),
    "country": paises_unicos
})

# Agregar clave limpia de país para facilitar validaciones
df_country_dimension["country_key"] = df_country_dimension["country"].apply(lambda x: crear_clave_texto(x))

df_country_dimension.head()

,country_id,country,country_key
0,1,Afghanistan,afghanistan
1,2,Albania,albania
2,3,Algeria,algeria
3,4,Andorra,andorra
4,5,Angola,angola


#### Explicación del identificador `country_id`

El identificador `country_id` representa de forma única a cada país encontrado en las fuentes transformadas. Este ID será útil para relacionar incidentes, indicadores de ciberseguridad y datos de usuarios de internet sin depender únicamente del nombre textual del país.

Esto ayuda a organizar los datos como si se estuviera preparando una estructura relacional, donde `df_country_dimension` funcionaría como una tabla de referencia o dimensión.

### 6.2 Asignación de identificadores a los DataFrames principales

Luego se agregan identificadores secuenciales a cada DataFrame principal. Estos identificadores permiten reconocer de forma única cada registro dentro de su respectiva fuente.

Se crean los siguientes identificadores:

- `threat_id` para los incidentes de ciberseguridad.
- `security_index_id` para los registros de índices de ciberseguridad.
- `internet_user_id` para los registros de usuarios de internet por país.

In [161]:
# Agregar country_id al dataset de amenazas y crear threat_id
df_threats_identified = df_threats_transformed.merge(
    df_country_dimension[["country_id", "country"]],
    on="country",
    how="left"
)

df_threats_identified.insert(0, "threat_id", range(1, len(df_threats_identified) + 1))

# Agregar country_id al dataset de índices de ciberseguridad y crear security_index_id
df_cyber_security_identified = df_cyber_security_transformed.merge(
    df_country_dimension[["country_id", "country"]],
    on="country",
    how="left"
)

df_cyber_security_identified.insert(0, "security_index_id", range(1, len(df_cyber_security_identified) + 1))

# Agregar country_id al dataset de usuarios de internet y crear internet_user_id
df_internet_users_identified = df_internet_users_transformed.merge(
    df_country_dimension[["country_id", "country"]],
    on="country",
    how="left"
)

df_internet_users_identified.insert(0, "internet_user_id", range(1, len(df_internet_users_identified) + 1))

print("Identificadores agregados correctamente a los DataFrames principales.")

Identificadores agregados correctamente a los DataFrames principales.


#### Explicación de los identificadores de registros

Los identificadores `threat_id`, `security_index_id` e `internet_user_id` permiten distinguir cada fila de forma única dentro de su fuente. Esto es útil para auditoría, trazabilidad, relaciones entre tablas y futuras cargas en PostgreSQL.

Además, al agregar `country_id` en cada DataFrame, se facilita la relación entre los datasets mediante un identificador numérico en lugar de depender solo del nombre del país.

### 6.3 Creación de una vista enriquecida con identificadores

Finalmente, se genera una nueva vista integrada llamada `df_threats_enriched_identified`. Esta vista conserva los identificadores creados y mantiene la información contextual de ciberseguridad e internet por país.

Esta estructura resulta útil para análisis posteriores porque cada incidente tiene un `threat_id`, un `country_id` y variables complementarias relacionadas con indicadores digitales y de ciberseguridad.

In [162]:
# Recrear la vista enriquecida tomando como base el DataFrame de amenazas con identificadores
df_threats_enriched_identified = (
    df_threats_identified
    .merge(cyber_security_context, on="country", how="left")
    .merge(internet_users_context, on="country", how="left")
)

# Reordenar columnas principales para facilitar lectura
columnas_inicio = ["threat_id", "country_id", "country", "year", "attack_type"]
columnas_restantes = [col for col in df_threats_enriched_identified.columns if col not in columnas_inicio]
df_threats_enriched_identified = df_threats_enriched_identified[columnas_inicio + columnas_restantes]

df_threats_enriched_identified.head()

,threat_id,country_id,country,year,attack_type,target_industry,financial_loss_in_million,number_of_affected_users,attack_source,security_vulnerability_type,defense_mechanism_used,incident_resolution_time_in_hours,country_key,country_year_key,financial_loss_usd,incident_year_start_date,financial_loss_category,affected_users_category,resolution_time_category,security_region,cei,gci,ncsi,ddl,cei_percentage,gci_level,ncsi_level,ddl_level,available_security_indicators,has_complete_security_index,subregion,internet_region,internet_data_year,internet_users,population_2021,percentage_internet_users,internet_users_millions,population_millions,internet_users_ratio,internet_penetration_level
0,1,42,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63,china,china_2019,80530000.0,2019-01-01,Crítica,Impacto crítico,Resolución lenta,Asia-Pacific,0.483,92.53,51.95,62.41,48.3,Muy alto,Medio,Medio,4,True,Eastern Asia,Asia,2022,1102140000,1425893465,77.3,1102.14,1425.89,0.773,Alta
1,2,42,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71,china,china_2019,62190000.0,2019-01-01,Alta,Impacto medio,Resolución lenta,Asia-Pacific,0.483,92.53,51.95,62.41,48.3,Muy alto,Medio,Medio,4,True,Eastern Asia,Asia,2022,1102140000,1425893465,77.3,1102.14,1425.89,0.773,Alta
2,3,92,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20,india,india_2017,38650000.0,2017-01-01,Media,Impacto alto,Resolución rápida,Asia-Pacific,0.597,97.50,59.74,40.02,59.7,Muy alto,Medio,Medio,4,True,Southern Asia,Asia,2024,881250000,1407563842,62.6,881.25,1407.56,0.626,Media
3,4,211,United Kingdom,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7,united_kingdom,united_kingdom_2024,41440000.0,2024-01-01,Media,Impacto alto,Resolución rápida,Europe,0.207,99.54,89.61,79.96,20.7,Muy alto,Muy alto,Alto,4,True,Northern Europe,Europe,2021,65001016,67281039,96.6,65.00,67.28,0.966,Muy alta
4,5,76,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68,germany,germany_2018,74410000.0,2018-01-01,Alta,Impacto crítico,Resolución lenta,Europe,0.241,97.41,90.91,80.01,24.1,Muy alto,Muy alto,Alto,4,True,Western Europe,Europe,2021,77794405,83408554,93.3,77.79,83.41,0.933,Muy alta


### 6.4 Validación de identificadores generados

Se valida que los identificadores sean únicos y que la cantidad de registros se mantenga correctamente después de asignarlos. Esta revisión confirma que cada tabla puede ser utilizada de forma organizada en futuras etapas de análisis o carga.

In [163]:
validacion_identificadores = pd.DataFrame([
    {
        "dataframe": "df_country_dimension",
        "identificador": "country_id",
        "filas": df_country_dimension.shape[0],
        "ids_unicos": df_country_dimension["country_id"].nunique(),
        "es_unico": df_country_dimension["country_id"].is_unique
    },
    {
        "dataframe": "df_threats_identified",
        "identificador": "threat_id",
        "filas": df_threats_identified.shape[0],
        "ids_unicos": df_threats_identified["threat_id"].nunique(),
        "es_unico": df_threats_identified["threat_id"].is_unique
    },
    {
        "dataframe": "df_cyber_security_identified",
        "identificador": "security_index_id",
        "filas": df_cyber_security_identified.shape[0],
        "ids_unicos": df_cyber_security_identified["security_index_id"].nunique(),
        "es_unico": df_cyber_security_identified["security_index_id"].is_unique
    },
    {
        "dataframe": "df_internet_users_identified",
        "identificador": "internet_user_id",
        "filas": df_internet_users_identified.shape[0],
        "ids_unicos": df_internet_users_identified["internet_user_id"].nunique(),
        "es_unico": df_internet_users_identified["internet_user_id"].is_unique
    }
])

validacion_identificadores

,dataframe,identificador,filas,ids_unicos,es_unico
0,df_country_dimension,country_id,222,222,True
1,df_threats_identified,threat_id,3000,3000,True
2,df_cyber_security_identified,security_index_id,192,192,True
3,df_internet_users_identified,internet_user_id,215,215,True


In [164]:
# Validar que no existan registros sin country_id después de la relación con la dimensión de países
validacion_country_id = pd.DataFrame([
    {
        "dataframe": "df_threats_identified",
        "registros_sin_country_id": df_threats_identified["country_id"].isnull().sum()
    },
    {
        "dataframe": "df_cyber_security_identified",
        "registros_sin_country_id": df_cyber_security_identified["country_id"].isnull().sum()
    },
    {
        "dataframe": "df_internet_users_identified",
        "registros_sin_country_id": df_internet_users_identified["country_id"].isnull().sum()
    }
])

validacion_country_id

,dataframe,registros_sin_country_id
0,df_threats_identified,0
1,df_cyber_security_identified,0
2,df_internet_users_identified,0


### Hallazgos de la generación de identificadores

Durante esta fase se creó una dimensión de países llamada `df_country_dimension`, la cual contiene un identificador numérico secuencial `country_id` para cada país presente en las fuentes transformadas. Este identificador permite relacionar las fuentes de forma más ordenada y reduce la dependencia del texto del país como clave de unión.

También se generaron identificadores únicos para cada fuente principal: `threat_id` para los incidentes de ciberseguridad, `security_index_id` para los indicadores de ciberseguridad e `internet_user_id` para los datos de usuarios de internet.

Estos identificadores facilitan la trazabilidad de los registros, la organización de los DataFrames y una posible carga posterior en una base de datos relacional. Además, permiten que la información pueda estructurarse en tablas relacionadas, siguiendo una lógica similar a un modelo dimensional o relacional.

Finalmente, se creó la vista `df_threats_enriched_identified`, que conserva los identificadores y combina los incidentes de ciberseguridad con información contextual de indicadores de seguridad e internet. Esto permite continuar el análisis con una base integrada, organizada y más fácil de relacionar.